In [1]:
from upxo.ggrowth.mcgs import mcgs
from scipy.spatial import cKDTree
import numpy as np
import pyvista as pv
from copy import deepcopy
import matplotlib.pyplot as plt
import upxo._sup.data_ops as DO
import upxo._sup.dataTypeHandlers as dth

In [2]:
input_dashboard='C:\\Development\\UPXO\\upxo_library\\src\\upxo\\demos\\gsgen\\demo3d1.xls'

In [3]:
"""Instantiate ythe mcgs class"""
# pxt = mcgs(study='independent', input_dashboard='demo_3d_01.xls')
pxt = mcgs(study='independent', input_dashboard=input_dashboard)

C:\Development\UPXO\upxo_library\src\upxo\interfaces\user_inputs
C:\Development\UPXO\upxo_library\src\upxo\demos\gsgen\demo3d1.xls
Algo_hops details
(('300a', 100),)
[False]


Let's look at the data structure a bit.

In [4]:
pxt

UPXO 3D.MCGS
(A: GRID)::   x:(0.0,15.0,1.0),   y:(0.0,15.0,1.0),   z:(0.0,15.0,1.0)
(B: SIMPAR)::   nstates: 3  mcsteps: 10  algorithms: (('300a', 100),)
(C: MESHPAR)::   GB Conformity: Non-Conformal
               Target FE Software: Abaqus  Element type: quad4
------------------------------------------------------------

In [5]:
pxt.uigrid

Attribues of gridding definitions: 
     TYPE: square
     DIMENSIONALITY: 3
     X: (0.0, 15.0, 1.0)
     Y: (0.0, 15.0, 1.0)
     Z: (0.0, 15.0, 1.0)
     PIXEL SIZE: 1.0
     TRANSFORMATION: none

In [6]:
pxt.uisim

Attributes of Simulation parameters:
     MCSTEPS: 10
     S: 3 - will be deprecated.
     STATE SAMPLING SCHEME: rejection
     CONSIDER BOLTZMANN PROBABILITY: False
     S BOLTZAMNN PROBABILITY: [1. 1. 1.]
     MAXIMUM BOLTZMANN TEMPERATURE FACTOR: 0.0
     BOUNDARY CONDITION TYPE: wrapped
     NON LOCALITY: 1
     KINETICITY: static

In [7]:
pxt.vox_size

1.0

In [8]:
pxt.vox_length

1.0

Simulate the grain structuer and inspect the data-structure.

In [9]:
pxt.simulate(verbose=False)


 Initiating Monte-Carlo simulation
     xmin, xmax, xinc: 0.0, 15.0, 1.0
     ymin, ymax, yinc: 0.0, 15.0, 1.0
     zmin, zmax, zinc: 0.0, 15.0, 1.0
     No. of states: 3
     Dimensionality: 3
Using ALG-300a
////////////////////////////////
Initiating grain growth
----------------------------------------
GS temporal slice 0 stored
GS temporal slice 1 stored
GS temporal slice 2 stored
GS temporal slice 3 stored
GS temporal slice 4 stored
GS temporal slice 5 stored
GS temporal slice 6 stored
GS temporal slice 7 stored
GS temporal slice 8 stored
GS temporal slice 9 stored
|--------------- MC SIM RUN COMPLETED on: ALG310---------------|
Number of gs tslices: 10


Sinulation has created temporally evolved grain structure. These are
contained in pxt.gs. We will now inspect this and provide details of it below.

In [10]:
print(pxt.gs.keys(), '\n')
print(f"First temporal slice is ---> {pxt.gs[0]}\n", 20*'-  ')

dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]) 

First temporal slice is ---> UPXO. gs-tslice.3d. 2982128908464
 -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  


This is a dictionatry containing monte-carlo time step as keys and the corresponding UPXO grain structue objects as the value.

Thoushgh grain struvcture have been created, these are not fully populated with the information needed for the completion of either conformal meshing pipeline or the representativeness qualification pipeline or any other major thing for that matter. We will achivve these in steps and at the complettion of each step, the necessary data will be populated and enable user to perform certain functyions / operations. We will not, at this moment look at all details of the this object, but rather look at its details in steps below.

Lets extract a slice (say 5th) from the grain structue temporal stack

In [11]:
tslice = pxt.m[-1]
gstslice = pxt.gs[tslice]
print(gstslice)

UPXO. gs-tslice.3d. 2982240289056


In [12]:
gstslice.char_morphology_of_grains(label_str_order=3,
                                   find_grain_voxel_locs=True,
                                   find_spatial_bounds_of_grains=True,
                                   force_compute=True)
gstslice.set_mprops(volnv=True, eqdia=True, eqdia_base_size_spec='volnv',
                    arbbox=False, arbbox_fmt='gid_dict',
                    arellfit=False, arellfit_metric='max', arellfit_calculate_efits=True,
                    arellfit_efit_routine=1, arellfit_efit_regularize_data=True,
                    solidity=False, sol_nan_treatment='replace', sol_inf_treatment='replace',
                    sol_nan_replacement=-1, sol_inf_replacement=-1,
                    sanv=False, sanv_N=26, rat_sanv_volnv=False, sanv_verbosity=1E2, disp_msg='')

---------------------------------------- 
Finding grains.
No. of grains detected = 4
---------------------------------------- 
Setting PyVista grid.
---------------------------------------- 
Adding scalar field: lgi to PyVista grid self.pvgrid.
Access: self.pvgrid.cell_data['lgi']

Finding voxel locations of grains in lgi.

Finding normal and extended spatial bounds of all grains.
Completed ----------------------------------------
----------------------------------------
---------------------------------------- 
Setting grain volumes (metric: 'volnv') -> 
Grain volumes (metric: 'volnv') -> : have been set.
 ----------------------------------------
---------------------------------------- 
Setting grain volumes (metric: 'volnv') -> 
Grain volumes (metric: 'volnv') -> : have been set.
 ----------------------------------------
---------------------------------------- 
Setting grain eq.sph.dia. values (metric: 'eqdia').


All moorphologuical properti3es can be accessed as below.

In [13]:
print(gstslice.mprop.keys())

dict_keys(['volnv', 'volsr', 'volch', 'sanv', 'savi', 'sasr', 'psa', 'pernv', 'pervl', 'pergl', 'eqdia', 'feqdia', 'kx', 'ky', 'kz', 'ksr', 'arbbox', 'arellfit', 'sol', 'ecc', 'com', 'sph', 'fn', 'rnd', 'mi', 'fdim', 'rat_sanv_volnv'])


The keys and thier ecxplanations as of 15-08-2024 is proided below:
- volnv: Volume by number of voxels
- volsr: Volume after grain boundary surface reconstruction
- volch: Volume of convex hull
- sanv: surface area by number of voxels
- savi: surface area by voxel interfaces
- sasr: surface area after grain boundary surface reconstruction
- pernv: perimeter by number of voxels
- pervl: perimeter by voxel edge lines
- pergl: perimeter by geometric grain boundary line segments
- eqdia: eqvivalent diameter
- kx: grain boundary voxel local curvature in yz plane
- ky: grain boundary voxel local curvature in xz plane
- kz: grain boundary voxel local curvature in xy plane
- kxyz: mean(kx, ky, kz)
- ksr: k computed from surface reconstruction.
- arbbox: aspect ratio by bounding box
- arefit: aspect ratio by ellipsoidal fit
- sol: solidity
- ecc: eccentricity - how much the shape of the grain differs from a sphere.
- com: compactness
- sph: sphericity
- fn: flatness
- rnd: roundness
- mi: moment of inertia tensor
- fdim: fractal dimension

We can use the following to extract specific grain structure IDs.

In [14]:
gstslice.get_largest_gids()

array([3])

In [15]:
len(gstslice.get_smallest_gids())

2

In [16]:
len(gstslice.single_voxel_grains)

0

In [17]:
len(gstslice.small_grains(vth=2))  # vth: Volume Threshold

0

In [18]:
len(gstslice.large_grains(vth=2))

4

In [19]:
len(gstslice.find_grains_by_nvoxels(nvoxels=2))

0

The following properties can be used to querry specific morphologuical properties.

In [20]:
low = gstslice.smallest_volume
high = gstslice.largest_volume
prop_range = high-low

gstslice.find_grains_by_mprop_range(prop_name='volnv', 
                                    low=low+prop_range*0.25,
                                    high=high+prop_range*0.75,
                                    low_ineq='ge',
                                    high_ineq='le')

array([1, 3])

You can use gstslice.gpos to know the relative locations of grains in the grain structure position. It is a nested dictionary

In [21]:
print(gstslice.gpos)

{}


We see that this is inexistent. This is because, it was not caculated using characterisation. We will do it now

In [22]:
# gstslice.set_grain_positions(verbose=False)

In [23]:
# gstslice.gpos.keys()

In [24]:
# print(gstslice.gpos['face']['left'])

Let us extract values of a scalar variable along a stright line, with the
straight line being specified by starting and ending coordinate indice
locatoins.

This returns a numpy array of lgi values. The line between the two
specified end points is generated using Bresenham algorithm in 3D.

In [25]:
gstslice.get_values_along_line([0, 0, 0], [9, 9, 9], scalars='lgi')

array([1, 3, 3, 3, 3, 3, 3, 3, 1, 1], dtype=int32)

You can also calculate the intercept - grain size in a couple of different
ways.

In [26]:
gstslice.get_igs_properties_along_line([0, 0, 0], [9, 9, 9], scalars='lgi')

{'ng': 2,
 'nv': array([3, 7]),
 'igs': np.float64(5.0),
 'igs_median': np.float64(5.0),
 'igs_range': np.int64(4),
 'igs_std': np.float64(2.0),
 'igs_var': np.float64(4.0),
 'sv': array([1, 3, 3, 3, 3, 3, 3, 3, 1, 1], dtype=int32),
 'sv_unique': array([1, 3], dtype=int32)}

This gives a dictionary with the following keys:
* ng: Number of grains
* nv: np array of number ofvoxels between all grain boundaris on the line
* igs: intercept grain size (mean)
* igs_median: median value of the igs
* igs_range: range of grain sizes along the line
* igs_std: standard deviation of the of grain sizes along the line
* igs_var: variance of the of grain sizes along the line
* sv: scalar values along the line between the two specified locaytions
* sv_unique: unique scalar values along the line between the two specified locaytions

In [27]:
gstslice.get_igs_along_line([0, 0, 0], [9, 9, 9], metric='mean', minimum=True,
                       maximum=True, std=True, variance=True, verbose=True)

---------------------------------------- 
Getting intercept grain size @line: [0, 0, 0]---[9, 9, 9].


{'igs': np.float64(5.0),
 'metric': 'mean',
 'min': np.int64(3),
 'max': np.int64(7),
 'std': np.float64(2.0),
 'var': np.float64(4.0)}

This gives us a dictionaryu of following keys:
- 'igs': value of the metric of igs values.
- 'metric': metric specified by the user.
- 'min': Minimum value. Key only present if minimum is specified True.
- 'max': Maximum value. Key only present if maximum is specified True.
- 'std': Std. deviation value. Key only present if std is specified True.
- 'var': Variance value. Key only present if variance is specified True.

In [28]:
gstslice.get_igs_along_lines(metric='mean', minimum=True, maximum=True,
                             std=True, variance=True, lines_gen_method=1,
                             lines_kwargs1={'plane': 'z',
                                            'start_skip1': 0,
                                            'start_skip2': 0,
                                            'incr1': 10, 'incr2': 10,
                                            'inclination': 'none',
                                            'inclination_extent': 0,
                                            'shift_seperately': False,
                                            'shift_starts': False,
                                            'shift_ends': False,
                                            'start_shift': 0, 'end_shift': 0})

---------------------------------------- 
Getting intercept grain size @line: [0 0 0]---[14  0  0].
---------------------------------------- 
Getting intercept grain size @line: [ 0  0 10]---[14  0 10].
---------------------------------------- 
Getting intercept grain size @line: [ 0 10  0]---[14 10  0].
---------------------------------------- 
Getting intercept grain size @line: [ 0 10 10]---[14 10 10].


{'igs': np.float64(7.5),
 'metric': 'mean',
 'min': array([7, 6, 4, 5]),
 'max': array([ 8,  9, 11, 10]),
 'std': array([0.5, 1.5, 3.5, 2.5]),
 'var': array([ 0.25,  2.25, 12.25,  6.25]),
 'igs_all': array([7.5, 7.5, 7.5, 7.5]),
 'ngrains': 4}

This produces a dictionary with the folloiwng keys
* 'igs': overall intercept grain size of the grain structuer.
* 'metric': metric used for calulation of the overall igs value.
* 'min': minimum values of igs in each sampling line.
* 'max': maximum values of igs in each sampling line.
* 'std': standard deviation of igs values in each sampling line.
* 'var': varianbce of igs values in each sampline line.
* 'igs_all': individsual igs over each sampling line.
* 'ngrains': total number of grains in grain structure.

In [29]:
gstslice.igs_sed_ratio(metric='mean', lines_gen_method=1,
                       reset_grain_size=True, base_size_spec='volnv',
                       lines_kwargs1={'plane': 'z',
                                      'start_skip1': 0, 'start_skip2': 0,
                                      'incr1': 3, 'incr2': 3,
                                      'inclination': 'random',
                                      'inclination_extent': 0,
                                      'shift_seperately': False,
                                      'shift_starts': False,
                                      'shift_ends': False,
                                      'start_shift': 0, 'end_shift': 0})

---------------------------------------- 
Getting intercept grain size @line: [ 0 12  9]---[14  0  9].
---------------------------------------- 
Getting intercept grain size @line: [0 3 9]---[14  0  3].
---------------------------------------- 
Getting intercept grain size @line: [ 0  9 12]---[14 12  0].
---------------------------------------- 
Getting intercept grain size @line: [ 0 12  0]---[14  0  0].
---------------------------------------- 
Getting intercept grain size @line: [0 9 0]---[14  3  9].
---------------------------------------- 
Getting intercept grain size @line: [ 0 12  3]---[14  3  3].
---------------------------------------- 
Getting intercept grain size @line: [0 0 6]---[14  6  9].
---------------------------------------- 
Getting intercept grain size @line: [0 6 3]---[14 12  6].
---------------------------------------- 
Getting intercept grain size @line: [0 6 0]---[14  0 12].
---------------------------------------- 
Getting intercept grain size @line: [0 9 9]---

np.float64(1.1376638075050653)

In [30]:
"A = gstslice.extract_subdomains_random(p=5, q=5, r=5, n=2, feature_name='base', user_fid=None, make_pvgrids=True)"

"A = gstslice.extract_subdomains_random(p=5, q=5, r=5, n=2, feature_name='base', user_fid=None, make_pvgrids=True)"

In [31]:
"A.keys()"

'A.keys()'

In [32]:
"len(A['sd'])"

"len(A['sd'])"

In [33]:
"A['pvgrids'][0]"

"A['pvgrids'][0]"

In [34]:
"A['pvgrids'][0].plot()"

"A['pvgrids'][0].plot()"

In [35]:
gstslice.n

4

In [36]:
"""gstslice.clean_gs_GMD_by_source_erosion_v1(prop='volnv', threshold=40,
                                           parameter_metric='mean',
                                           reset_pvgrid_every_iter=False,
                                           find_neigh_every_iter=False,
                                           find_grvox_every_iter=False,
                                           find_grspabnds_every_iter=False)"""

"gstslice.clean_gs_GMD_by_source_erosion_v1(prop='volnv', threshold=40,\n                                           parameter_metric='mean',\n                                           reset_pvgrid_every_iter=False,\n                                           find_neigh_every_iter=False,\n                                           find_grvox_every_iter=False,\n                                           find_grspabnds_every_iter=False)"

In [37]:
gstslice.n

4

In [38]:
gstslice.viz_mesh_slice_ortho(scalar='lgi', cmap='viridis', style='surface', throw=False, pvp=None)

Widget(value='<iframe src="http://localhost:59359/index.html?ui=P_0x2b65a0878c0_0&reconnect=auto" class="pyvis…

In [39]:
import cc3d
lfi = cc3d.connected_components(gstslice.lfi, connectivity=6)

In [40]:
import os
directory = r"C:\Development\zmesh\data"
file_path = os.path.join(directory, "UPXO_mcgs3d_01.npy")
# 2. Ensure the directory exists (creates it if missing)
os.makedirs(directory, exist_ok=True)
# 3. Save the array
np.save(file_path, lfi)